# 2.1 — Convex Sets & Functions

Convexity is the geometric promise that straight-line mixtures behave well: feasible mixtures stay feasible, and function values along a mixture never rise above the straight chord between endpoint values. In this lesson, you will build convex sets, convex functions, Jensen checks, tangent lower bounds, and curvature certificates from scratch with NumPy so later optimization guarantees feel mechanical rather than magical.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build convexity one idea at a time. Run each cell in order and inspect the printed numbers and plots — every inequality is checked directly, and every visualization is chosen to reveal the geometry. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

▶ What you'll see: the only tools used in this lesson are NumPy for arrays and Matplotlib for visual inspection.

### 1. Convex combinations and convex sets

A convex set is a region that contains the whole straight segment between any two of its points. Algebraically, the segment is generated by convex combinations: $z=\lambda x+(1-\lambda)y$ with $0\le\lambda\le1$. The weights are nonnegative and sum to one, so the new point is an average, not an extrapolation.

In [ ]:
x_w = np.array([0.0, 0.0])
y_w = np.array([2.0, 1.0])
lam_w = 0.25
z_w = lam_w * x_w + (1 - lam_w) * y_w

print("x:", x_w, "y:", y_w, "lambda:", lam_w)
print("convex combination z:", z_w)

assert np.allclose(z_w, [1.5, 0.75])

▶ What you'll see: the weighted average lands at `[1.5, 0.75]`, between the two endpoints.

In [ ]:
square_min_w = np.array([0.0, 0.0])
square_max_w = np.array([2.0, 2.0])
inside_square_w = np.all((z_w >= square_min_w) & (z_w <= square_max_w))

print("inside [0,2]^2?", inside_square_w)

assert inside_square_w

▶ What you'll see: the mixture remains inside the square, which is the feasibility-preservation property.

In [ ]:
lams_w = np.linspace(0, 1, 25)
segment_w = np.array([lam * x_w + (1 - lam) * y_w for lam in lams_w])
plt.figure(figsize=(4.4, 3.4))
plt.plot([0, 2, 2, 0, 0], [0, 0, 2, 2, 0], color="black", label="square boundary")
plt.plot(segment_w[:, 0], segment_w[:, 1], color="teal", marker=".", label="all mixtures")
plt.scatter([x_w[0], y_w[0], z_w[0]], [x_w[1], y_w[1], z_w[1]], color=["green", "orange", "red"])
plt.xlim(-0.2, 2.2); plt.ylim(-0.2, 2.2); plt.gca().set_aspect("equal")
plt.title("1: segment stays inside a convex square")
plt.legend(); plt.show()

▶ What you'll see: every dot on the straight segment stays inside the square.

*Why it's done this way:* Convex combinations are the smallest algebraic test for “straight-line safety.” Optimization algorithms repeatedly average, interpolate, and step along lines; if the constraint set is closed under those mixtures, local movement between feasible points cannot accidentally leave the feasible region.

### 2. Nonconvex sets fail by one missing midpoint

To prove a set is not convex, we do not need to check every pair. One counterexample is enough: find two feasible points whose midpoint is infeasible. A ring is the classic example — connected, but with a hole where a straight chord can leave the set.

In [ ]:
p_w = np.array([1.0, 0.0])
q_w = np.array([-1.0, 0.0])
mid_w = 0.5 * p_w + 0.5 * q_w
radius_mid_w = np.linalg.norm(mid_w)

print("p:", p_w, "q:", q_w, "midpoint:", mid_w)
print("midpoint radius:", radius_mid_w)

assert np.allclose(mid_w, [0.0, 0.0])

▶ What you'll see: opposite points on a ring average to the origin.

In [ ]:
inner_w, outer_w = 0.7, 1.3
p_in_ring_w = inner_w <= np.linalg.norm(p_w) <= outer_w
q_in_ring_w = inner_w <= np.linalg.norm(q_w) <= outer_w
mid_in_ring_w = inner_w <= radius_mid_w <= outer_w

print("endpoints in ring?", p_in_ring_w, q_in_ring_w)
print("midpoint in ring?", mid_in_ring_w)

assert p_in_ring_w and q_in_ring_w and not mid_in_ring_w

▶ What you'll see: both endpoints are feasible, but their midpoint falls in the hole.

In [ ]:
theta_w = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(4, 4))
plt.fill(outer_w * np.cos(theta_w), outer_w * np.sin(theta_w), color="lightgray")
plt.fill(inner_w * np.cos(theta_w), inner_w * np.sin(theta_w), color="white")
plt.plot([p_w[0], q_w[0]], [p_w[1], q_w[1]], color="crimson", linewidth=2, label="chord")
plt.scatter([p_w[0], q_w[0], mid_w[0]], [p_w[1], q_w[1], mid_w[1]], color=["green", "green", "red"])
plt.gca().set_aspect("equal"); plt.title("2: a ring is connected but not convex")
plt.legend(); plt.show()

▶ What you'll see: the chord cuts through the empty center, so connectedness is not enough.

*Why it's done this way:* Convexity is a universal statement over all pairs and all weights, so one violating mixture disproves it. The midpoint is the simplest mixture to test because both weights are equal and the geometry is easiest to see.

### 3. Convex functions and Jensen's inequality

A function is convex when the graph lies below every chord between two graph points. The algebraic form is $f(\lambda x+(1-\lambda)y)\le\lambda f(x)+(1-\lambda)f(y)$. For $f(x)=x^2$, this says the square of an average is no larger than the average of the squares.

In [ ]:
x1_w, x2_w = 1.0, 3.0
lam_j_w = 0.25
mix_w = lam_j_w * x1_w + (1 - lam_j_w) * x2_w
left_w = mix_w ** 2
right_w = lam_j_w * x1_w ** 2 + (1 - lam_j_w) * x2_w ** 2

print("mixed input:", mix_w)
print("f(mixed input):", left_w)
print("mixed function values:", right_w)

assert left_w == 6.25 and right_w == 7.0 and left_w <= right_w

▶ What you'll see: `6.25 <= 7.0`, the Jensen inequality for this pair.

In [ ]:
grid_w = np.linspace(0, 4, 200)
fgrid_w = grid_w ** 2
chord_x_w = np.array([x1_w, x2_w])
chord_y_w = chord_x_w ** 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_w, fgrid_w, color="teal", label="f(x)=x²")
plt.plot(chord_x_w, chord_y_w, color="black", linestyle="--", label="chord")
plt.scatter([mix_w], [left_w], color="green", label="f(mix)")
plt.scatter([mix_w], [right_w], color="red", label="chord at mix")
plt.title("3: convex graph sits below the chord")
plt.legend(); plt.show()

▶ What you'll see: the green point on the curve is below the red point on the chord.

*Why it's done this way:* We compare `f` after averaging inputs against averaging the endpoint outputs because optimization constantly replaces hard landscapes with linear interpolation. A convex function rewards averaging: uncertainty or mixing in the input cannot make the function value worse than the corresponding average output.

### 4. Curvature and tangent lower bounds certify convexity

For smooth one-dimensional functions, nonnegative second derivative is a local curvature certificate for convexity. The same geometry says every tangent line is a global lower bound: $f(x)\ge f(x_0)+f'(x_0)(x-x_0)$. That tangent-support property is why gradients later certify global optima in convex optimization.

In [ ]:
x0_w = 2.0
x_test_w = 3.0
f0_w = x0_w ** 2
grad0_w = 2 * x0_w
tangent_at_test_w = f0_w + grad0_w * (x_test_w - x0_w)
f_test_w = x_test_w ** 2

print("tangent at x=3:", tangent_at_test_w)
print("f(3):", f_test_w)

assert tangent_at_test_w == 8.0 and f_test_w == 9.0 and f_test_w >= tangent_at_test_w

▶ What you'll see: the function value 9 sits above the tangent value 8.

In [ ]:
xs_w = np.linspace(-1, 5, 200)
tangent_w = f0_w + grad0_w * (xs_w - x0_w)
plt.figure(figsize=(4.4, 3.2))
plt.plot(xs_w, xs_w ** 2, color="teal", label="f(x)=x²")
plt.plot(xs_w, tangent_w, color="crimson", linestyle="--", label="tangent at x0=2")
plt.scatter([x0_w, x_test_w], [f0_w, f_test_w], color="black")
plt.title("4: tangent is a global lower bound")
plt.legend(); plt.show()

▶ What you'll see: the dashed line touches the bowl at `x0=2` and stays below it everywhere shown.

In [ ]:
H_w = np.array([[2.0, 0.0], [0.0, 6.0]])
eigs_w = np.linalg.eigvalsh(H_w)

print("Hessian eigenvalues:", eigs_w)
print("positive semidefinite?", np.all(eigs_w >= 0))

assert np.allclose(eigs_w, [2.0, 6.0])

▶ What you'll see: both Hessian eigenvalues are positive, so the quadratic bowl curves upward in every direction.

*Why it's done this way:* A second derivative or Hessian checks curvature locally, while the tangent inequality turns that local curvature into a global support statement. Positive Hessian eigenvalues mean every direction has upward curvature, so no direction can hide a local valley that is not globally consistent with the bowl.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Convex combinations preserve a box

A convex combination is a weighted average. If both endpoints live in a box, every coordinate of
that average stays between the same coordinate-wise bounds.

In [ ]:
import numpy as np                              # arrays and numerical checks.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # -> seed 0

print("rng seed:", 0)                           # -> 0

t1_x = np.array([0.0, 2.0])                     # -> [0.0, 2.0]

print("x:", t1_x.tolist())                      # -> [0.0, 2.0]

t1_y = np.array([4.0, 0.0])                     # -> [4.0, 0.0]

print("y:", t1_y.tolist())                      # -> [4.0, 0.0]

t1_lam = 0.25                                   # -> 0.25

print("lambda:", t1_lam)                        # -> 0.25

t1_weight_x = t1_lam * t1_x                     # -> [0.0, 0.5]

print("lambda * x:", t1_weight_x.tolist())      # -> [0.0, 0.5]

t1_weight_y = (1.0 - t1_lam) * t1_y             # -> [3.0, 0.0]

print("(1-lambda) * y:", t1_weight_y.tolist())  # -> [3.0, 0.0]

t1_mix = t1_weight_x + t1_weight_y              # -> [3.0, 0.5]

print("mixture:", t1_mix.tolist())              # -> [3.0, 0.5]

t1_lower = np.array([0.0, 0.0])                 # -> [0.0, 0.0]

print("box lower:", t1_lower.tolist())          # -> [0.0, 0.0]

t1_upper = np.array([4.0, 2.0])                 # -> [4.0, 2.0]

print("box upper:", t1_upper.tolist())          # -> [4.0, 2.0]

t1_inside_each = (t1_mix >= t1_lower) & (t1_mix <= t1_upper)  # -> [True, True]

print("coordinate checks:", t1_inside_each.tolist())         # -> [True, True]

t1_inside = bool(np.all(t1_inside_each))         # -> True

print("inside box?", t1_inside)                 # -> True

t1_lams = np.linspace(0.0, 1.0, 6)              # -> [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

print("lambda grid:", np.round(t1_lams, 1).tolist())          # -> [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

t1_segment = t1_lams[:, None] * t1_x + (1.0 - t1_lams[:, None]) * t1_y  # -> [[4.0, 0.0], [3.2, 0.4], [2.4, 0.8], [1.6, 1.2], [0.8, 1.6], [0.0, 2.0]]

print("segment points:", np.round(t1_segment, 2).tolist())              # -> [[4.0, 0.0], [3.2, 0.4], [2.4, 0.8], [1.6, 1.2], [0.8, 1.6], [0.0, 2.0]]

assert t1_inside and np.all((t1_segment >= t1_lower) & (t1_segment <= t1_upper))

plt.figure(figsize=(4.4, 3.2))
plt.plot([0, 4, 4, 0, 0], [0, 0, 2, 2, 0], color="black", label="box")
plt.plot(t1_segment[:, 0], t1_segment[:, 1], marker="o", color="teal", label="mixtures")
plt.scatter([t1_x[0], t1_y[0], t1_mix[0]], [t1_x[1], t1_y[1], t1_mix[1]], color=["green", "orange", "crimson"], zorder=3)
plt.title("Toy 1 · mixtures stay in the box")
plt.xlabel("coordinate 1")
plt.ylabel("coordinate 2")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

▶ What you'll see: the red mixture `[3.0, 0.5]` lies on the segment and inside the black box.

### ✍️ Toy 2 · One midpoint can disprove convexity

To show a set is nonconvex, find two feasible endpoints whose midpoint is infeasible. A ring fails
because opposite points average into the missing center.

In [ ]:
import numpy as np                              # arrays and norms.

t2_rng = np.random.default_rng(0)               # -> seed 0

print("rng seed:", 0)                           # -> 0

t2_p = np.array([1.0, 0.0])                     # -> [1.0, 0.0]

print("p:", t2_p.tolist())                      # -> [1.0, 0.0]

t2_q = np.array([-1.0, 0.0])                    # -> [-1.0, 0.0]

print("q:", t2_q.tolist())                      # -> [-1.0, 0.0]

t2_mid = 0.5 * t2_p + 0.5 * t2_q                # -> [0.0, 0.0]

print("midpoint:", t2_mid.tolist())             # -> [0.0, 0.0]

t2_inner = 0.6                                  # -> 0.6

print("inner radius:", t2_inner)                # -> 0.6

t2_outer = 1.4                                  # -> 1.4

print("outer radius:", t2_outer)                # -> 1.4

t2_points = np.vstack([t2_p, t2_q, t2_mid])     # -> [[1.0, 0.0], [-1.0, 0.0], [0.0, 0.0]]

print("points checked:", t2_points.tolist())    # -> [[1.0, 0.0], [-1.0, 0.0], [0.0, 0.0]]

t2_radii = np.linalg.norm(t2_points, axis=1)    # -> [1.0, 1.0, 0.0]

print("radii:", t2_radii.tolist())              # -> [1.0, 1.0, 0.0]

t2_feasible = (t2_inner <= t2_radii) & (t2_radii <= t2_outer)  # -> [True, True, False]

print("in ring?", t2_feasible.tolist())         # -> [True, True, False]

t2_counterexample = bool(t2_feasible[0] and t2_feasible[1] and not t2_feasible[2])  # -> True

print("counterexample?", t2_counterexample)     # -> True

t2_theta = np.linspace(0.0, 2.0 * np.pi, 9)     # -> [0.0, 0.785, 1.571, 2.356, 3.142, 3.927, 4.712, 5.498, 6.283]

print("theta grid:", np.round(t2_theta, 3).tolist())           # -> [0.0, 0.785, 1.571, 2.356, 3.142, 3.927, 4.712, 5.498, 6.283]

t2_outer_xy = np.column_stack([t2_outer * np.cos(t2_theta), t2_outer * np.sin(t2_theta)])  # -> ring outer points

print("outer ring points:", np.round(t2_outer_xy, 3).tolist())  # -> [[1.4, 0.0], [0.99, 0.99], [0.0, 1.4], [-0.99, 0.99], [-1.4, 0.0], [-0.99, -0.99], [-0.0, -1.4], [0.99, -0.99], [1.4, -0.0]]

t2_inner_xy = np.column_stack([t2_inner * np.cos(t2_theta), t2_inner * np.sin(t2_theta)])  # -> ring inner points

print("inner ring points:", np.round(t2_inner_xy, 3).tolist())  # -> [[0.6, 0.0], [0.424, 0.424], [0.0, 0.6], [-0.424, 0.424], [-0.6, 0.0], [-0.424, -0.424], [-0.0, -0.6], [0.424, -0.424], [0.6, -0.0]]

assert t2_counterexample

plt.figure(figsize=(4, 4))
plt.plot(t2_outer_xy[:, 0], t2_outer_xy[:, 1], color="gray", label="outer edge")
plt.plot(t2_inner_xy[:, 0], t2_inner_xy[:, 1], color="gray", linestyle="--", label="hole edge")
plt.plot([t2_p[0], t2_q[0]], [t2_p[1], t2_q[1]], color="crimson", label="midpoint chord")
plt.scatter([t2_p[0], t2_q[0], t2_mid[0]], [t2_p[1], t2_q[1], t2_mid[1]], color=["green", "green", "red"], zorder=3)
plt.title("Toy 2 · midpoint falls in the hole")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

▶ What you'll see: the two green endpoints are in the ring, but the red midpoint is in the hole.

### ✍️ Toy 3 · Jensen compares two ways to mix

For a convex function, evaluating after mixing inputs is no larger than mixing the endpoint values.
Here `x²` makes the Jensen gap visible with one pair of numbers.

In [ ]:
import numpy as np                              # arrays and scalar arithmetic.

t3_rng = np.random.default_rng(0)               # -> seed 0

print("rng seed:", 0)                           # -> 0

t3_x = 1.0                                      # -> 1.0

print("x:", t3_x)                               # -> 1.0

t3_y = 5.0                                      # -> 5.0

print("y:", t3_y)                               # -> 5.0

t3_lam = 0.25                                   # -> 0.25

print("lambda:", t3_lam)                        # -> 0.25

t3_mix = t3_lam * t3_x + (1.0 - t3_lam) * t3_y  # -> 4.0

print("mixed input:", t3_mix)                   # -> 4.0

t3_fx = t3_x ** 2                               # -> 1.0

print("f(x):", t3_fx)                            # -> 1.0

t3_fy = t3_y ** 2                               # -> 25.0

print("f(y):", t3_fy)                            # -> 25.0

t3_left = t3_mix ** 2                           # -> 16.0

print("f(mixed input):", t3_left)               # -> 16.0

t3_right = t3_lam * t3_fx + (1.0 - t3_lam) * t3_fy  # -> 19.0

print("mixed function values:", t3_right)       # -> 19.0

t3_gap = t3_right - t3_left                     # -> 3.0

print("Jensen gap:", t3_gap)                    # -> 3.0

t3_grid = np.linspace(0.0, 6.0, 7)              # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]

print("plot grid:", t3_grid.tolist())           # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]

t3_curve = t3_grid ** 2                         # -> [0.0, 1.0, 4.0, 9.0, 16.0, 25.0, 36.0]

print("f(grid):", t3_curve.tolist())            # -> [0.0, 1.0, 4.0, 9.0, 16.0, 25.0, 36.0]

t3_chord_x = np.array([t3_x, t3_y])             # -> [1.0, 5.0]

print("chord x:", t3_chord_x.tolist())          # -> [1.0, 5.0]

t3_chord_y = t3_chord_x ** 2                    # -> [1.0, 25.0]

print("chord y:", t3_chord_y.tolist())          # -> [1.0, 25.0]

assert t3_left <= t3_right and t3_gap == 3.0

plt.figure(figsize=(4.5, 3.1))
plt.plot(t3_grid, t3_curve, color="teal", marker="o", label="f(x)=x²")
plt.plot(t3_chord_x, t3_chord_y, color="black", linestyle="--", label="endpoint chord")
plt.scatter([t3_mix, t3_mix], [t3_left, t3_right], color=["green", "red"], zorder=3)
plt.title("Toy 3 · Jensen gap")
plt.xlabel("input")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the green curve value `16.0` sits below the red chord value `19.0`.

### ✍️ Toy 4 · Tangent lower bounds sit below a convex curve

For a differentiable convex function, the tangent line supports the curve from below. The gap is
zero at the tangent point and nonnegative at every checked point.

In [ ]:
import numpy as np                              # arrays and tangent arithmetic.

t4_rng = np.random.default_rng(0)               # -> seed 0

print("rng seed:", 0)                           # -> 0

t4_x0 = 2.0                                     # -> 2.0

print("tangent point:", t4_x0)                  # -> 2.0

t4_test = 4.0                                   # -> 4.0

print("test point:", t4_test)                   # -> 4.0

t4_f0 = t4_x0 ** 2                              # -> 4.0

print("f(x0):", t4_f0)                           # -> 4.0

t4_grad = 2.0 * t4_x0                           # -> 4.0

print("gradient at x0:", t4_grad)               # -> 4.0

t4_tangent_test = t4_f0 + t4_grad * (t4_test - t4_x0)  # -> 12.0

print("tangent at test:", t4_tangent_test)      # -> 12.0

t4_f_test = t4_test ** 2                        # -> 16.0

print("f(test):", t4_f_test)                    # -> 16.0

t4_gap = t4_f_test - t4_tangent_test            # -> 4.0

print("test gap:", t4_gap)                      # -> 4.0

t4_grid = np.linspace(0.0, 5.0, 6)              # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

print("plot grid:", t4_grid.tolist())           # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

t4_curve = t4_grid ** 2                         # -> [0.0, 1.0, 4.0, 9.0, 16.0, 25.0]

print("f(grid):", t4_curve.tolist())            # -> [0.0, 1.0, 4.0, 9.0, 16.0, 25.0]

t4_tangent = t4_f0 + t4_grad * (t4_grid - t4_x0)  # -> [-4.0, 0.0, 4.0, 8.0, 12.0, 16.0]

print("tangent grid:", t4_tangent.tolist())     # -> [-4.0, 0.0, 4.0, 8.0, 12.0, 16.0]

t4_gaps = t4_curve - t4_tangent                 # -> [4.0, 1.0, 0.0, 1.0, 4.0, 9.0]

print("curve minus tangent:", t4_gaps.tolist()) # -> [4.0, 1.0, 0.0, 1.0, 4.0, 9.0]

assert np.all(t4_gaps >= 0.0) and t4_gap == 4.0

plt.figure(figsize=(4.5, 3.1))
plt.plot(t4_grid, t4_curve, marker="o", color="teal", label="f(x)=x²")
plt.plot(t4_grid, t4_tangent, marker="s", color="crimson", linestyle="--", label="tangent")
plt.scatter([t4_x0], [t4_f0], color="black", zorder=3)
plt.title("Toy 4 · tangent stays below")
plt.xlabel("x")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the dashed tangent touches at `x=2` and stays below all sampled curve points.

### ✍️ Toy 5 · Hessian eigenvalues certify convex curvature

A quadratic with a positive semidefinite Hessian curves upward in every direction. Eigenvalues give
a finite curvature check for that infinitely many-direction statement.

In [ ]:
import numpy as np                              # arrays and eigenvalues.

t5_rng = np.random.default_rng(0)               # -> seed 0

print("rng seed:", 0)                           # -> 0

t5_H = np.array([[4.0, 1.0], [1.0, 2.0]])       # -> [[4.0, 1.0], [1.0, 2.0]]

print("Hessian:", t5_H.tolist())                # -> [[4.0, 1.0], [1.0, 2.0]]

t5_eigs = np.linalg.eigvalsh(t5_H)              # -> [1.58578644, 4.41421356]

print("eigenvalues:", np.round(t5_eigs, 3).tolist())           # -> [1.586, 4.414]

t5_dir1 = np.array([1.0, 0.0])                  # -> [1.0, 0.0]

print("direction 1:", t5_dir1.tolist())         # -> [1.0, 0.0]

t5_dir2 = np.array([0.0, 1.0])                  # -> [0.0, 1.0]

print("direction 2:", t5_dir2.tolist())         # -> [0.0, 1.0]

t5_dir3 = np.array([1.0, 1.0]) / np.sqrt(2.0)   # -> [0.707, 0.707]

print("direction 3:", np.round(t5_dir3, 3).tolist())            # -> [0.707, 0.707]

t5_curv1 = float(t5_dir1 @ t5_H @ t5_dir1)      # -> 4.0

print("curvature direction 1:", t5_curv1)       # -> 4.0

t5_curv2 = float(t5_dir2 @ t5_H @ t5_dir2)      # -> 2.0

print("curvature direction 2:", t5_curv2)       # -> 2.0

t5_curv3 = float(t5_dir3 @ t5_H @ t5_dir3)      # -> 4.0

print("curvature direction 3:", round(t5_curv3, 3))             # -> 4.0

t5_curvatures = np.array([t5_curv1, t5_curv2, t5_curv3])         # -> [4.0, 2.0, 4.0]

print("sample curvatures:", np.round(t5_curvatures, 3).tolist()) # -> [4.0, 2.0, 4.0]

t5_positive = bool(np.all(t5_eigs > 0.0))        # -> True

print("positive definite?", t5_positive)         # -> True

assert t5_positive and np.all(t5_curvatures > 0.0)

plt.figure(figsize=(4.4, 2.8))
plt.bar(["λ1", "λ2"], t5_eigs, color=["seagreen", "teal"])
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 5 · positive Hessian eigenvalues")
plt.ylabel("curvature")
plt.show()

▶ What you'll see: both eigenvalue bars are above zero, so the quadratic has upward curvature.


## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Build a convex combination

**Goal.** Average two vectors with nonnegative weights that sum to one, because convexity is about straight-line mixtures.

In [ ]:
x_b1 = np.array([0.0, 0.0])
y_b1 = np.array([2.0, 1.0])
lam_b1 = 0.25
z_b1 = lam_b1 * x_b1 + (1 - lam_b1) * y_b1

print("z_b1:", z_b1)

assert np.allclose(z_b1, [1.5, 0.75])

▶ What you'll see: the combination is a point between the endpoints.

In [ ]:
seg_b1 = np.array([lam * x_b1 + (1 - lam) * y_b1 for lam in np.linspace(0, 1, 25)])
plt.figure(figsize=(4.2, 3.2))
plt.plot(seg_b1[:, 0], seg_b1[:, 1], color="teal", marker=".", label="convex combinations")
plt.scatter([x_b1[0], y_b1[0], z_b1[0]], [x_b1[1], y_b1[1], z_b1[1]], color=["green", "orange", "red"], zorder=3)
plt.title("Basic 1: weighted average on the segment")
plt.gca().set_aspect("equal"); plt.legend(); plt.show()

▶ What you'll see: the red weighted average lies directly on the segment between the two endpoints.

👀 Takeaway: convex combinations are weighted averages, not arbitrary linear combinations.

### Basic 2 — Check that a square is convex for one segment

**Goal.** Verify that a segment between two points in `[0,2]^2` stays inside the square.

In [ ]:
x_b2 = np.array([0.2, 1.8])
y_b2 = np.array([1.8, 0.4])
lams_b2 = np.linspace(0, 1, 11)
pts_b2 = np.array([lam * x_b2 + (1 - lam) * y_b2 for lam in lams_b2])
inside_b2 = np.all((pts_b2 >= 0) & (pts_b2 <= 2))

print("sampled points:\n", np.round(pts_b2, 2))
print("all inside square?", inside_b2)

assert inside_b2

▶ What you'll see: every sampled point has both coordinates between 0 and 2.

In [ ]:
plt.figure(figsize=(4.2, 3.4))
plt.fill([0, 2, 2, 0], [0, 0, 2, 2], color="lightblue", alpha=0.35, label="[0,2]²")
plt.plot([0, 2, 2, 0, 0], [0, 0, 2, 2, 0], color="black")
plt.plot(pts_b2[:, 0], pts_b2[:, 1], color="teal", marker="o", label="sampled segment")
plt.scatter([x_b2[0], y_b2[0]], [x_b2[1], y_b2[1]], color=["green", "orange"], zorder=3)
plt.xlim(-0.2, 2.2); plt.ylim(-0.2, 2.2); plt.gca().set_aspect("equal")
plt.title("Basic 2: segment stays in the square")
plt.legend(); plt.show()

▶ What you'll see: every sampled mixture lies inside the shaded square.

👀 Takeaway: boxes are convex because coordinate-wise averages remain between coordinate-wise bounds.

### Basic 3 — See a nonconvex midpoint failure

**Goal.** Use two points in a ring to show that connected does not imply convex.

In [ ]:
p_b3 = np.array([1.0, 0.0])
q_b3 = np.array([-1.0, 0.0])
mid_b3 = 0.5 * (p_b3 + q_b3)
inner_b3, outer_b3 = 0.7, 1.3
mid_ok_b3 = inner_b3 <= np.linalg.norm(mid_b3) <= outer_b3

print("midpoint:", mid_b3, "in ring?", mid_ok_b3)

assert not mid_ok_b3

▶ What you'll see: the midpoint is the origin, which lies in the ring's hole.

In [ ]:
theta_b3 = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(4, 4))
plt.fill(outer_b3 * np.cos(theta_b3), outer_b3 * np.sin(theta_b3), color="lightgray")
plt.fill(inner_b3 * np.cos(theta_b3), inner_b3 * np.sin(theta_b3), color="white")
plt.plot([p_b3[0], q_b3[0]], [p_b3[1], q_b3[1]], color="crimson", linewidth=2, label="segment")
plt.scatter([p_b3[0], q_b3[0], mid_b3[0]], [p_b3[1], q_b3[1], mid_b3[1]], color=["green", "green", "red"], zorder=3)
plt.gca().set_aspect("equal"); plt.title("Basic 3: midpoint falls in the hole")
plt.legend(); plt.show()

▶ What you'll see: the red midpoint is outside the ring even though both endpoints are inside it.

👀 Takeaway: one infeasible midpoint is enough to disprove set convexity.

### Basic 4 — Plot a convex segment inside a disk

**Goal.** Visualize that the disk contains the chord between two interior points.

In [ ]:
x_b4 = np.array([0.6, 0.2])
y_b4 = np.array([-0.3, 0.7])
lams_b4 = np.linspace(0, 1, 30)
seg_b4 = np.array([lam * x_b4 + (1 - lam) * y_b4 for lam in lams_b4])
radii_b4 = np.linalg.norm(seg_b4, axis=1)

print("max segment radius:", round(float(radii_b4.max()), 3))

assert radii_b4.max() <= 1.0
plt.figure(figsize=(4, 4))
t_b4 = np.linspace(0, 2 * np.pi, 200)
plt.plot(np.cos(t_b4), np.sin(t_b4), color="black")
plt.plot(seg_b4[:, 0], seg_b4[:, 1], color="teal")
plt.scatter([x_b4[0], y_b4[0]], [x_b4[1], y_b4[1]], color="orange")
plt.gca().set_aspect("equal"); plt.title("Basic 4: segment in unit disk"); plt.show()

▶ What you'll see: the segment lies completely inside the circular boundary.

👀 Takeaway: Euclidean balls are convex because averaging cannot increase distance beyond the endpoint bound.

### Basic 5 — Check Jensen for a square function

**Goal.** Compare `f` at an averaged input with the averaged function values.

In [ ]:
x_b5, y_b5, lam_b5 = 1.0, 3.0, 0.25
mix_b5 = lam_b5 * x_b5 + (1 - lam_b5) * y_b5
lhs_b5 = mix_b5 ** 2
rhs_b5 = lam_b5 * x_b5 ** 2 + (1 - lam_b5) * y_b5 ** 2

print("f(mix):", lhs_b5, "weighted f values:", rhs_b5)

assert lhs_b5 == 6.25 and rhs_b5 == 7.0 and lhs_b5 <= rhs_b5

▶ What you'll see: the convexity inequality holds numerically.

In [ ]:
grid_b5 = np.linspace(0, 4, 200)
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_b5, grid_b5 ** 2, color="teal", label="f(x)=x²")
plt.plot([x_b5, y_b5], [x_b5 ** 2, y_b5 ** 2], color="black", linestyle="--", label="chord")
plt.scatter([mix_b5, mix_b5], [lhs_b5, rhs_b5], color=["green", "red"], zorder=3)
plt.title("Basic 5: Jensen gap for x²")
plt.legend(); plt.show()

▶ What you'll see: the green curve value at the mixed input sits below the red chord value.

👀 Takeaway: convex functions put the function value at an average below the chord value.

### Basic 6 — Plot the chord above `x²`

**Goal.** Make the Jensen inequality visible on a graph.

In [ ]:
x_b6 = np.linspace(0, 4, 200)
y_b6 = x_b6 ** 2
end_x_b6 = np.array([1.0, 3.0])
end_y_b6 = end_x_b6 ** 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(x_b6, y_b6, label="x²", color="teal")
plt.plot(end_x_b6, end_y_b6, "--", label="chord", color="black")
plt.scatter(end_x_b6, end_y_b6, color="orange")
plt.title("Basic 6: chord above a convex graph")
plt.legend(); plt.show()

▶ What you'll see: the dashed chord stays above the curved graph between the endpoints.

👀 Takeaway: chord-above-graph is the geometric definition of convex functions.

### Basic 7 — Use a second derivative certificate

**Goal.** Confirm that `f(x)=x²` is convex by checking its curvature.

In [ ]:
xs_b7 = np.array([-2.0, 0.0, 2.0])
second_derivative_b7 = np.full_like(xs_b7, 2.0)

print("sample x values:", xs_b7)
print("f''(x):", second_derivative_b7)

assert np.all(second_derivative_b7 >= 0)

▶ What you'll see: the second derivative is positive at every sampled location.

In [ ]:
curve_x_b7 = np.linspace(-2.5, 2.5, 200)
plt.figure(figsize=(4.4, 3.2))
plt.plot(curve_x_b7, curve_x_b7 ** 2, color="teal", label="f(x)=x²")
plt.scatter(xs_b7, xs_b7 ** 2, c=second_derivative_b7, cmap="viridis", s=70, label="sampled curvature")
plt.title("Basic 7: positive curvature samples")
plt.colorbar(label="f''(x)")
plt.legend(); plt.show()

▶ What you'll see: the sampled points on the bowl are colored by the same positive curvature value.

👀 Takeaway: in one dimension, nonnegative second derivative is a smooth convexity certificate.

### Basic 8 — Check a tangent lower bound

**Goal.** Verify that a tangent line to a convex function stays below the function.

In [ ]:
x0_b8 = 2.0
x_b8 = np.array([1.0, 2.0, 3.0])
f_b8 = x_b8 ** 2
tangent_b8 = x0_b8 ** 2 + 2 * x0_b8 * (x_b8 - x0_b8)

print("f values:", f_b8)
print("tangent values:", tangent_b8)

assert np.all(f_b8 >= tangent_b8)

▶ What you'll see: the tangent matches at `x=2` and is lower at neighboring points.

In [ ]:
grid_b8 = np.linspace(0.5, 3.5, 200)
tangent_grid_b8 = x0_b8 ** 2 + 2 * x0_b8 * (grid_b8 - x0_b8)
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_b8, grid_b8 ** 2, color="teal", label="f(x)=x²")
plt.plot(grid_b8, tangent_grid_b8, color="crimson", linestyle="--", label="tangent at x=2")
plt.scatter(x_b8, f_b8, color="black", zorder=3)
plt.title("Basic 8: tangent lower bound")
plt.legend(); plt.show()

▶ What you'll see: the dashed tangent touches the curve once and stays below nearby function values.

👀 Takeaway: tangent lower bounds are the gradient-based signature of convexity.

### Basic 9 — Recognize a convex nonsmooth function

**Goal.** Inspect `|x|`, which is convex even though it has a corner at zero.

In [ ]:
x_b9 = np.linspace(-2, 2, 9)
f_b9 = np.abs(x_b9)
mid_b9 = 0.5 * (-1.0) + 0.5 * (1.0)
lhs_b9 = abs(mid_b9)
rhs_b9 = 0.5 * abs(-1.0) + 0.5 * abs(1.0)

print("|x| values:", f_b9)
print("midpoint Jensen check:", lhs_b9, "<=", rhs_b9)

assert lhs_b9 <= rhs_b9

▶ What you'll see: the V-shape satisfies Jensen even at the nondifferentiable corner.

In [ ]:
dense_b9 = np.linspace(-2, 2, 200)
plt.figure(figsize=(4.4, 3.2))
plt.plot(dense_b9, np.abs(dense_b9), color="teal", label="|x|")
plt.plot([-1, 1], [1, 1], color="black", linestyle="--", label="chord from -1 to 1")
plt.scatter([mid_b9], [lhs_b9], color="red", zorder=3, label="midpoint value")
plt.title("Basic 9: nonsmooth convex V-shape")
plt.legend(); plt.show()

▶ What you'll see: the V-shaped graph lies below the chord even at the sharp corner.

👀 Takeaway: convexity does not require smoothness; corners can still be globally bowl-shaped.

### Basic 10 — Check a Hessian's eigenvalues

**Goal.** Use positive eigenvalues to certify a two-dimensional quadratic bowl.

In [ ]:
H_b10 = np.array([[2.0, 0.0], [0.0, 6.0]])
eigs_b10 = np.linalg.eigvalsh(H_b10)

print("Hessian eigenvalues:", eigs_b10)

assert np.allclose(eigs_b10, [2.0, 6.0])
assert np.all(eigs_b10 >= 0)

▶ What you'll see: every curvature direction is nonnegative.

In [ ]:
x1_b10 = np.linspace(-2, 2, 80)
x2_b10 = np.linspace(-1.2, 1.2, 80)
X1_b10, X2_b10 = np.meshgrid(x1_b10, x2_b10)
Z_b10 = 0.5 * (H_b10[0, 0] * X1_b10 ** 2 + H_b10[1, 1] * X2_b10 ** 2)
plt.figure(figsize=(4.4, 3.6))
plt.contour(X1_b10, X2_b10, Z_b10, levels=12, cmap="viridis")
plt.scatter([0], [0], color="red", label="global minimum")
plt.title("Basic 10: PSD Hessian gives bowl contours")
plt.gca().set_aspect("equal"); plt.legend(); plt.show()

▶ What you'll see: nested ellipses surround one global minimum, matching the positive Hessian eigenvalues.

👀 Takeaway: a symmetric Hessian with nonnegative eigenvalues means a smooth quadratic is convex.

## 🟡 Easy

### Easy 1 — Test many mixtures in a triangle

**Goal.** Sample convex combinations of triangle vertices, because a simplex is the basic convex hull of finitely many points.

In [ ]:
V_e1 = np.array([[0.0, 0.0], [2.0, 0.0], [0.5, 1.5]])
weights_e1 = np.array([[0.2, 0.3, 0.5], [0.6, 0.1, 0.3], [0.1, 0.8, 0.1]])
pts_e1 = weights_e1 @ V_e1

print("weights row sums:", weights_e1.sum(axis=1))
print("triangle mixtures:\n", np.round(pts_e1, 3))

assert np.allclose(weights_e1.sum(axis=1), 1.0)

▶ What you'll see: each row of weights creates one point inside the triangle.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
closed_e1 = np.vstack([V_e1, V_e1[0]])
plt.plot(closed_e1[:, 0], closed_e1[:, 1], color="black")
plt.scatter(pts_e1[:, 0], pts_e1[:, 1], color="teal")
plt.gca().set_aspect("equal"); plt.title("Easy 1: convex hull of three vertices"); plt.show()

▶ What you'll see: all sampled mixtures lie inside the triangular hull.

👀 Takeaway: convex hull points are exactly nonnegative weighted averages whose weights sum to one.

### Easy 2 — Show a halfspace is convex

**Goal.** Verify that if two points satisfy `a·x <= b`, then their average does too.

In [ ]:
a_e2 = np.array([1.0, 2.0])
b_e2 = 5.0
x_e2 = np.array([1.0, 1.0])
y_e2 = np.array([3.0, 0.5])
lam_e2 = 0.4
z_e2 = lam_e2 * x_e2 + (1 - lam_e2) * y_e2
values_e2 = np.array([a_e2 @ x_e2, a_e2 @ y_e2, a_e2 @ z_e2])

print("a·x, a·y, a·z:", values_e2)

assert np.all(values_e2 <= b_e2)

▶ What you'll see: the mixed point still satisfies the linear inequality.

In [ ]:
xx_e2 = np.linspace(0, 5, 100)
yline_e2 = (b_e2 - xx_e2) / 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(xx_e2, yline_e2, color="black", label="a·x=b")
plt.fill_between(xx_e2, -0.2, yline_e2, color="lightblue", alpha=0.5, label="a·x<=b")
plt.scatter([x_e2[0], y_e2[0], z_e2[0]], [x_e2[1], y_e2[1], z_e2[1]], color=["green", "orange", "red"])
plt.ylim(-0.2, 3); plt.title("Easy 2: halfspace mixture"); plt.legend(); plt.show()

▶ What you'll see: the average point stays in the shaded feasible halfspace.

👀 Takeaway: linear inequalities define convex feasible regions because dot products preserve weighted averages.

### Easy 3 — Compare convex and concave chord tests

**Goal.** See that `x²` passes the chord test while `-x²` fails it for minimization.

In [ ]:
x_e3, y_e3, lam_e3 = -1.0, 2.0, 0.5
mix_e3 = lam_e3 * x_e3 + (1 - lam_e3) * y_e3
convex_lhs_e3 = mix_e3 ** 2
convex_rhs_e3 = lam_e3 * x_e3 ** 2 + (1 - lam_e3) * y_e3 ** 2
concave_lhs_e3 = -mix_e3 ** 2
concave_rhs_e3 = lam_e3 * (-x_e3 ** 2) + (1 - lam_e3) * (-y_e3 ** 2)

print("x² check:", convex_lhs_e3, "<=", convex_rhs_e3)
print("-x² check:", concave_lhs_e3, "<=", concave_rhs_e3)

assert convex_lhs_e3 <= convex_rhs_e3 and not (concave_lhs_e3 <= concave_rhs_e3)

▶ What you'll see: the bowl passes, but the upside-down bowl violates the convex inequality.

In [ ]:
grid_e3 = np.linspace(-2, 2, 200)
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_e3, grid_e3 ** 2, label="convex x²", color="teal")
plt.plot(grid_e3, -grid_e3 ** 2, label="nonconvex -x²", color="crimson")
plt.title("Easy 3: bowl versus upside-down bowl")
plt.legend(); plt.show()

▶ What you'll see: the convex bowl opens upward, while the concave curve opens downward.

👀 Takeaway: convexity is directional; `-x²` is good for maximization but nonconvex for minimization.

### Easy 4 — Verify Jensen for a weighted expectation

**Goal.** Treat weights as probabilities and check Jensen's inequality for `exp(x)`.

In [ ]:
values_e4 = np.array([-1.0, 0.0, 2.0])
probs_e4 = np.array([0.2, 0.5, 0.3])
mean_e4 = float(np.sum(probs_e4 * values_e4))
lhs_e4 = np.exp(mean_e4)
rhs_e4 = float(np.sum(probs_e4 * np.exp(values_e4)))

print("E[X]:", round(mean_e4, 3))
print("exp(E[X]):", round(lhs_e4, 3), "E[exp(X)]:", round(rhs_e4, 3))

assert lhs_e4 <= rhs_e4

▶ What you'll see: the exponential at the mean is below the probability-weighted exponential values.

In [ ]:
plt.figure(figsize=(4.4, 3.2))
plt.bar(["exp(E[X])", "E[exp(X)]"], [lhs_e4, rhs_e4], color=["teal", "orange"])
plt.title("Easy 4: Jensen as weighted averaging")
plt.ylabel("value"); plt.show()

▶ What you'll see: the Jensen upper bar is taller.

👀 Takeaway: expectations are convex combinations, so Jensen is convexity applied to probability weights.

### Easy 5 — Draw tangent lower bounds for a quadratic

**Goal.** Plot several tangent lines to see that each supports the same convex function from below.

In [ ]:
xs_e5 = np.linspace(-3, 3, 200)
f_e5 = xs_e5 ** 2
anchors_e5 = np.array([-2.0, 0.0, 2.0])
tangents_e5 = np.array([a ** 2 + 2 * a * (xs_e5 - a) for a in anchors_e5])
violations_e5 = np.max(tangents_e5 - f_e5)

print("largest tangent minus function:", round(float(violations_e5), 8))

assert violations_e5 <= 1e-10

▶ What you'll see: no tangent line rises above the quadratic.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.plot(xs_e5, f_e5, color="black", linewidth=2, label="x²")
for i_e5, a_e5 in enumerate(anchors_e5):
    plt.plot(xs_e5, tangents_e5[i_e5], linestyle="--", label=f"tangent {a_e5:g}")
plt.ylim(-2, 10); plt.title("Easy 5: tangent lower bounds")
plt.legend(); plt.show()

▶ What you'll see: every dashed tangent touches once and stays below the bowl.

👀 Takeaway: convex gradients define global lower bounds, not just local slopes.

## 🔴 Advanced

### Advanced 1 — Compare local minima in convex and nonconvex landscapes

**Goal.** Sample two one-dimensional functions to see why convexity rules out deceptive local minima.

In [ ]:
x_a1 = np.linspace(-3, 3, 601)
convex_a1 = (x_a1 - 0.5) ** 2
nonconvex_a1 = (x_a1 ** 2 - 1) ** 2 + 0.15 * x_a1
convex_min_x_a1 = x_a1[np.argmin(convex_a1)]
nonconvex_min_x_a1 = x_a1[np.argmin(nonconvex_a1)]

print("convex grid minimizer:", round(float(convex_min_x_a1), 3))
print("nonconvex best grid minimizer:", round(float(nonconvex_min_x_a1), 3))

assert abs(convex_min_x_a1 - 0.5) < 0.01

▶ What you'll see: the convex quadratic has one bottom, while the nonconvex curve has multiple valleys.

In [ ]:
plt.figure(figsize=(5, 3.2))
plt.plot(x_a1, convex_a1, label="convex", color="teal")
plt.plot(x_a1, nonconvex_a1, label="nonconvex", color="crimson")
plt.title("Advanced 1: convex bowl vs nonconvex valleys")
plt.legend(); plt.show()

▶ What you'll see: the red curve has separate basins, which can trap local methods.

👀 Takeaway: convex minimization is trustworthy because any local minimum is global.

### Advanced 2 — Certify a quadratic with Hessian eigenvalues

**Goal.** Build a quadratic form and classify it from its Hessian spectrum.

In [ ]:
H_a2 = np.array([[4.0, 1.0], [1.0, 3.0]])
b_a2 = np.array([1.0, -2.0])
eigs_a2 = np.linalg.eigvalsh(H_a2)
x_star_a2 = -np.linalg.solve(H_a2, b_a2)

print("H eigenvalues:", np.round(eigs_a2, 3))
print("stationary point:", np.round(x_star_a2, 3))

assert np.all(eigs_a2 > 0)

▶ What you'll see: positive eigenvalues certify a strictly convex quadratic with one stationary point.

In [ ]:
xg_a2 = np.linspace(-1.5, 1.5, 80)
yg_a2 = np.linspace(-1.5, 1.5, 80)
X_a2, Y_a2 = np.meshgrid(xg_a2, yg_a2)
Z_a2 = 0.5 * (H_a2[0, 0] * X_a2 ** 2 + 2 * H_a2[0, 1] * X_a2 * Y_a2 + H_a2[1, 1] * Y_a2 ** 2) + b_a2[0] * X_a2 + b_a2[1] * Y_a2
plt.figure(figsize=(4.6, 3.6))
plt.contour(X_a2, Y_a2, Z_a2, levels=18, cmap="viridis")
plt.scatter([x_star_a2[0]], [x_star_a2[1]], color="red", label="global minimizer")
plt.title("Advanced 2: convex quadratic contours")
plt.legend(); plt.show()

▶ What you'll see: elliptical contours surround the unique minimizer.

👀 Takeaway: positive definite Hessians turn stationarity into a global minimum for quadratics.

### Advanced 3 — Estimate curvature with finite differences

**Goal.** Approximate second derivatives numerically, because real objectives may be available only as function evaluations.

In [ ]:
def f_a3(x_a3):
    return np.log1p(np.exp(x_a3))

xs_a3 = np.linspace(-4, 4, 41)
h_a3 = 1e-2
second_fd_a3 = (f_a3(xs_a3 + h_a3) - 2 * f_a3(xs_a3) + f_a3(xs_a3 - h_a3)) / h_a3 ** 2

print("min finite-difference curvature:", round(float(second_fd_a3.min()), 6))
print("max finite-difference curvature:", round(float(second_fd_a3.max()), 6))

assert second_fd_a3.min() > 0

▶ What you'll see: the softplus function has positive estimated curvature everywhere sampled.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.plot(xs_a3, second_fd_a3, color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 3: numerical curvature of softplus")
plt.xlabel("x"); plt.ylabel("finite-difference f''(x)"); plt.show()

▶ What you'll see: the curvature curve stays above zero.

👀 Takeaway: sampled curvature checks do not prove convexity globally, but they are useful diagnostics for smooth losses.

### Advanced 4 — Project a point onto a convex set

**Goal.** Project onto the probability simplex, because constrained convex optimization often repeatedly returns points to a convex feasible set.

In [ ]:
v_a4 = np.array([0.7, -0.2, 1.5])
u_a4 = np.sort(v_a4)[::-1]
cssv_a4 = np.cumsum(u_a4) - 1
rho_candidates_a4 = u_a4 - cssv_a4 / (np.arange(len(v_a4)) + 1) > 0
rho_a4 = np.where(rho_candidates_a4)[0][-1]
theta_a4 = cssv_a4[rho_a4] / (rho_a4 + 1)
w_a4 = np.maximum(v_a4 - theta_a4, 0)

print("theta:", round(float(theta_a4), 3))
print("projected point:", np.round(w_a4, 3), "sum:", round(float(w_a4.sum()), 3))

assert np.all(w_a4 >= 0) and np.allclose(w_a4.sum(), 1.0)

▶ What you'll see: the infeasible vector becomes a nonnegative vector summing to one.

In [ ]:
plt.figure(figsize=(4.6, 3.0))
idx_a4 = np.arange(len(v_a4))
plt.bar(idx_a4 - 0.18, v_a4, width=0.36, label="original", color="crimson")
plt.bar(idx_a4 + 0.18, w_a4, width=0.36, label="projected", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 4: projection onto simplex")
plt.legend(); plt.show()

▶ What you'll see: negative mass is clipped and excess mass is redistributed to satisfy the simplex constraint.

👀 Takeaway: projections rely on convexity so the closest feasible point is well-defined and unique for squared distance.

### Advanced 5 — Separate a point from a convex ball with a tangent hyperplane

**Goal.** Construct a supporting line to the unit ball, because convex sets can be certified by linear inequalities at their boundary.

In [ ]:
boundary_a5 = np.array([1.0 / np.sqrt(2), 1.0 / np.sqrt(2)])
outside_a5 = np.array([1.2, 1.2])
normal_a5 = boundary_a5.copy()
boundary_value_a5 = normal_a5 @ boundary_a5
outside_value_a5 = normal_a5 @ outside_a5

print("support value at boundary:", round(float(boundary_value_a5), 3))
print("outside value:", round(float(outside_value_a5), 3))

assert np.allclose(boundary_value_a5, 1.0) and outside_value_a5 > 1.0

▶ What you'll see: the outside point violates the tangent halfspace inequality while the boundary point is tight.

In [ ]:
t_a5 = np.linspace(0, 2 * np.pi, 300)
line_x_a5 = np.linspace(-0.2, 1.4, 100)
line_y_a5 = (1 - normal_a5[0] * line_x_a5) / normal_a5[1]
plt.figure(figsize=(4.4, 4.0))
plt.plot(np.cos(t_a5), np.sin(t_a5), color="black", label="unit ball boundary")
plt.plot(line_x_a5, line_y_a5, "--", color="crimson", label="supporting line")
plt.scatter([boundary_a5[0], outside_a5[0]], [boundary_a5[1], outside_a5[1]], color=["green", "red"])
plt.gca().set_aspect("equal"); plt.xlim(-1.2, 1.5); plt.ylim(-1.2, 1.5)
plt.title("Advanced 5: tangent separates outside point")
plt.legend(); plt.show()

▶ What you'll see: the dashed line touches the ball once and leaves the outside point on the wrong side.

👀 Takeaway: supporting hyperplanes turn convex geometry into linear certificates.